### Building a RAG System with LangChain and FAISS
Introduction
Retrieval-Augmented Generation (RAG) combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's traning data, RAG:
<ol>
<li>Retrieves relevant documents from a knowledge base </li>
<li>Uses these docuemnts as context for the LLM</li>
<li>Generates responses based on both the retrieved context and the model's knowledge</li>
</ol>

FAISS

Link: https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.



In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# Load libraries
import os
from dotenv import load_dotenv
import numpy as np
from typing import List, Dict, Any, Optional
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Langchain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough, RunnableSequence
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Langchain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader

# Load environment variables
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")



### Data Ingestion and Processing

In [3]:
sample_docs = [
    Document(
        page_content="""
        The capital of France is Paris. 
        It is known for its rich history, culture, and landmarks such as the Eiffel Tower and the Louvre Museum.
        Paris is also famous for its cuisine, fashion, and art scene.
        Visitors can enjoy a variety of attractions, including the Notre-Dame Cathedral, Montmartre, and the Seine River.
        Food lovers can indulge in French pastries, gourmet dining, and vibrant markets.
        For more information, visit the official tourism website of Paris: https://en.parisinfo.com/
    """,
        metadata={"source": "About Paris", "page": 1, "topic": "Geography" }
    ),
    Document(
        page_content="""
        The largest planet in our solar system is Jupiter.
        It is a gas giant and has a strong magnetic field.
        Jupiter has at least 79 moons, including the four largest known as the Galilean moons: 
        Io, Europa, Ganymede, and Callisto.
    """,
        metadata={"source": "Solar System Facts", "page": 2, "topic": "Astronomy" }
    ),
    Document(
        page_content="""
        The chemical symbol for water is H2O.
        Water is essential for all known forms of life and covers about 71% of the Earth's surface.
        It has unique properties such as being a universal solvent and having a high specific heat capacity.
    """,
        metadata={"source": "Basic Chemistry", "page": 3, "topic": "Chemistry" }
    )
]

print("Sample documents loaded successfully.")
print(sample_docs)

Sample documents loaded successfully.
[Document(metadata={'source': 'About Paris', 'page': 1, 'topic': 'Geography'}, page_content='\n        The capital of France is Paris. \n        It is known for its rich history, culture, and landmarks such as the Eiffel Tower and the Louvre Museum.\n        Paris is also famous for its cuisine, fashion, and art scene.\n        Visitors can enjoy a variety of attractions, including the Notre-Dame Cathedral, Montmartre, and the Seine River.\n        Food lovers can indulge in French pastries, gourmet dining, and vibrant markets.\n        For more information, visit the official tourism website of Paris: https://en.parisinfo.com/\n    '), Document(metadata={'source': 'Solar System Facts', 'page': 2, 'topic': 'Astronomy'}, page_content='\n        The largest planet in our solar system is Jupiter.\n        It is a gas giant and has a strong magnetic field.\n        Jupiter has at least 79 moons, including the four largest known as the Galilean moons: \

### Text Splitting

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=len, separators=[" "])
chunks = text_splitter.split_documents(sample_docs)
print("Documents split into chunks:")
print(f"Created {len(chunks)} chunks from the original documents.")
print(f"Content: {chunks[0].page_content} | Metadata: {chunks[0].metadata}")
print(f"Content: {chunks[1].page_content} | Metadata: {chunks[1].metadata}")
print(f"Content: {chunks[2].page_content} | Metadata: {chunks[2].metadata}")



Documents split into chunks:
Created 4 chunks from the original documents.
Content: The capital of France is Paris. 
        It is known for its rich history, culture, and landmarks such as the Eiffel Tower and the Louvre Museum.
        Paris is also famous for its cuisine, fashion, and art scene.
        Visitors can enjoy a variety of attractions, including the Notre-Dame Cathedral, Montmartre, and the Seine River.
        Food lovers can indulge in French pastries, gourmet dining, and vibrant markets.
        For more information, visit the official tourism website | Metadata: {'source': 'About Paris', 'page': 1, 'topic': 'Geography'}
Content: information, visit the official tourism website of Paris: https://en.parisinfo.com/ | Metadata: {'source': 'About Paris', 'page': 1, 'topic': 'Geography'}
Content: The largest planet in our solar system is Jupiter.
        It is a gas giant and has a strong magnetic field.
        Jupiter has at least 79 moons, including the four largest know

### Load the Embedding Models

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)
print("Embeddings model initialized successfully.")

# Example of creating a embeddings vector for a single text
sample_text="What is machine learning"
sample_embedding = embeddings.embed_query(sample_text)
print(f"Sample text: {sample_text}")
print(f"Sample embedding vector (first 5 dimensions): {sample_embedding[:5]}")

# list of texts to embed
texts = ["AI", "Machine Learning", "Deep Learning", "Natural Language Processing"]
batch_embeddings = embeddings.embed_documents(texts)
print("Batch embeddings created successfully.")
for text, embedding in zip(texts, batch_embeddings):
    print(f"Text: {text} | Embedding (first 5 dimensions): {embedding[:5]}")

Embeddings model initialized successfully.
Sample text: What is machine learning
Sample embedding vector (first 5 dimensions): [-0.005901336669921875, -0.00595855712890625, 0.0005350112915039062, -0.0335693359375, 0.0211944580078125]
Batch embeddings created successfully.
Text: AI | Embedding (first 5 dimensions): [-0.0081634521484375, -0.0246124267578125, 0.0029850006103515625, 0.0251617431640625, 0.006565093994140625]
Text: Machine Learning | Embedding (first 5 dimensions): [-0.022064208984375, -0.003543853759765625, -0.0191650390625, -0.034088134765625, 0.03375244140625]
Text: Deep Learning | Embedding (first 5 dimensions): [0.018035888671875, -0.036041259765625, -0.01104736328125, -9.298324584960938e-05, 0.0290069580078125]
Text: Natural Language Processing | Embedding (first 5 dimensions): [-0.044158935546875, 0.029144287109375, 0.02252197265625, -0.0234375, -0.0026035308837890625]


### Compare Embedding using cosine similarity

In [6]:

def compare_embeddings(text1: str, text2: str):
    """Compare the similarity between two texts using cosine similarity of their embeddings."""
    """
    Cosine similarity measures the angle between two vectors.
    - Result close to 1: Very similar
    - Result close to 0: Not related
    - Result close to -1: Opposite meanings
    """

    vec1 = embeddings.embed_query(text1)
    vec2 = embeddings.embed_query(text2)

    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1) # get the magnitude of vector A
    norm_vec2 = np.linalg.norm(vec2) # get the magnitude of vector B

    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0

    return dot_product / (norm_vec1 * norm_vec2)

# Test Semantic similarity
print("Comparing 'AI' and 'Machine Learning':")
similarity_score = compare_embeddings("AI", "Machine Learning")
print(f"Cosine Similarity Score: {similarity_score:.4f}")
print("Comparing 'AI' and 'Artificial Intelligence':")
similarity_score = compare_embeddings("AI", "Aritificial Intelligence")
print(f"Cosine Similarity Score: {similarity_score:.4f}")
print("Comparing 'AI' and 'Pizza':")
similarity_score = compare_embeddings("AI", "Pizza")
print(f"Cosine Similarity Score: {similarity_score:.4f}")

Comparing 'AI' and 'Machine Learning':
Cosine Similarity Score: 0.3724
Comparing 'AI' and 'Artificial Intelligence':
Cosine Similarity Score: 0.5364
Comparing 'AI' and 'Pizza':
Cosine Similarity Score: 0.2535


### Create FIASS Vector Store

In [7]:
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
print("FAISS vector store created successfully.")
print(f"Number of vectors in the store: {vectorstore.index.ntotal}")

FAISS vector store created successfully.
Number of vectors in the store: 4


In [8]:
vectorstore

### Save Vector Store for later use

In [9]:
vectorstore.save_local("faiss_index")
print("FAISS index saved locally as 'faiss_index'.")

FAISS index saved locally as 'faiss_index'.


### Load Vector Store

In [10]:
loaded_vectorstore = FAISS.load_local("faiss_index", 
                                      embeddings, 
                                      allow_dangerous_deserialization=True)
print("FAISS index loaded successfully from 'faiss_index'.")
print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors.")

FAISS index loaded successfully from 'faiss_index'.
Loaded vector store contains 4 vectors.


### Perform Similarity Search

In [11]:
query="What is the capital of France?"
results=vectorstore.similarity_search(query, k=3)
print(f"Top 3 results for query: '{query}'")
for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Content: {result.page_content}")
    print(f"Metadata: {result.metadata}")

Top 3 results for query: 'What is the capital of France?'
Result 1:
Content: The capital of France is Paris. 
        It is known for its rich history, culture, and landmarks such as the Eiffel Tower and the Louvre Museum.
        Paris is also famous for its cuisine, fashion, and art scene.
        Visitors can enjoy a variety of attractions, including the Notre-Dame Cathedral, Montmartre, and the Seine River.
        Food lovers can indulge in French pastries, gourmet dining, and vibrant markets.
        For more information, visit the official tourism website
Metadata: {'source': 'About Paris', 'page': 1, 'topic': 'Geography'}
Result 2:
Content: information, visit the official tourism website of Paris: https://en.parisinfo.com/
Metadata: {'source': 'About Paris', 'page': 1, 'topic': 'Geography'}
Result 3:
Content: The largest planet in our solar system is Jupiter.
        It is a gas giant and has a strong magnetic field.
        Jupiter has at least 79 moons, including the four lar

### Similarity Search with Score

In [12]:
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)
print(f"Top 3 results with similarity scores for query: '{query}'")
for result, score in results_with_scores:
    print(f"Similarity Score: {score:.4f}")
    print(f"Content: {result.page_content}")
    print(f"Metadata: {result.metadata}")


Top 3 results with similarity scores for query: 'What is the capital of France?'
Similarity Score: 0.5774
Content: The capital of France is Paris. 
        It is known for its rich history, culture, and landmarks such as the Eiffel Tower and the Louvre Museum.
        Paris is also famous for its cuisine, fashion, and art scene.
        Visitors can enjoy a variety of attractions, including the Notre-Dame Cathedral, Montmartre, and the Seine River.
        Food lovers can indulge in French pastries, gourmet dining, and vibrant markets.
        For more information, visit the official tourism website
Metadata: {'source': 'About Paris', 'page': 1, 'topic': 'Geography'}
Similarity Score: 1.1315
Content: information, visit the official tourism website of Paris: https://en.parisinfo.com/
Metadata: {'source': 'About Paris', 'page': 1, 'topic': 'Geography'}
Similarity Score: 1.7891
Content: The largest planet in our solar system is Jupiter.
        It is a gas giant and has a strong magnetic 

### Search with Metadata filtering

In [13]:
filter_dict = {"topic": "Geography"}
results = vectorstore.similarity_search(query, k=3, filter=filter_dict)
print(f"Top 3 results for query: '{query}' with filter: {filter_dict}")
for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Content: {result.page_content}")
    print(f"Metadata: {result.metadata}")

len(results)

Top 3 results for query: 'What is the capital of France?' with filter: {'topic': 'Geography'}
Result 1:
Content: The capital of France is Paris. 
        It is known for its rich history, culture, and landmarks such as the Eiffel Tower and the Louvre Museum.
        Paris is also famous for its cuisine, fashion, and art scene.
        Visitors can enjoy a variety of attractions, including the Notre-Dame Cathedral, Montmartre, and the Seine River.
        Food lovers can indulge in French pastries, gourmet dining, and vibrant markets.
        For more information, visit the official tourism website
Metadata: {'source': 'About Paris', 'page': 1, 'topic': 'Geography'}
Result 2:
Content: information, visit the official tourism website of Paris: https://en.parisinfo.com/
Metadata: {'source': 'About Paris', 'page': 1, 'topic': 'Geography'}


2

### Build RAG Chain with LCEL

In [14]:
# One way to initialize ChatOpenAI model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", 
                 temperature=0.2,
                 max_tokens=500)

text_response = llm.invoke("What is Large Lanaguage Models?")
text_response

AIMessage(content="Large Language Models (LLMs) are a type of artificial intelligence model that is trained on vast amounts of text data to understand and generate human language. These models are designed to process and generate natural language text, such as articles, essays, and conversations, with a high degree of accuracy and fluency. LLMs have been used in a variety of applications, including language translation, text generation, and sentiment analysis. Some popular examples of LLMs include OpenAI's GPT-3 and Google's BERT.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 15, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DM1k58R

In [15]:
from langchain_core.runnables import Runnable
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retriever
retriever.invoke("Tell me about France?")

[Document(id='087ec829-c945-41a0-841e-c5245e39af76', metadata={'source': 'About Paris', 'page': 1, 'topic': 'Geography'}, page_content='The capital of France is Paris. \n        It is known for its rich history, culture, and landmarks such as the Eiffel Tower and the Louvre Museum.\n        Paris is also famous for its cuisine, fashion, and art scene.\n        Visitors can enjoy a variety of attractions, including the Notre-Dame Cathedral, Montmartre, and the Seine River.\n        Food lovers can indulge in French pastries, gourmet dining, and vibrant markets.\n        For more information, visit the official tourism website'),
 Document(id='ab48d3d5-c6b4-4f0f-a5ce-231116ef4410', metadata={'source': 'About Paris', 'page': 1, 'topic': 'Geography'}, page_content='information, visit the official tourism website of Paris: https://en.parisinfo.com/'),
 Document(id='8d20c583-0be7-4bb4-8bf1-f87b5beea1c4', metadata={'source': 'Solar System Facts', 'page': 2, 'topic': 'Astronomy'}, page_content

In [16]:
query

'What is the capital of France?'

In [17]:
results_scores=vectorstore.similarity_search_with_score(query, k=3)
results_scores

[(Document(id='087ec829-c945-41a0-841e-c5245e39af76', metadata={'source': 'About Paris', 'page': 1, 'topic': 'Geography'}, page_content='The capital of France is Paris. \n        It is known for its rich history, culture, and landmarks such as the Eiffel Tower and the Louvre Museum.\n        Paris is also famous for its cuisine, fashion, and art scene.\n        Visitors can enjoy a variety of attractions, including the Notre-Dame Cathedral, Montmartre, and the Seine River.\n        Food lovers can indulge in French pastries, gourmet dining, and vibrant markets.\n        For more information, visit the official tourism website'),
  0.57744163),
 (Document(id='ab48d3d5-c6b4-4f0f-a5ce-231116ef4410', metadata={'source': 'About Paris', 'page': 1, 'topic': 'Geography'}, page_content='information, visit the official tourism website of Paris: https://en.parisinfo.com/'),
  1.1314671),
 (Document(id='8d20c583-0be7-4bb4-8bf1-f87b5beea1c4', metadata={'source': 'Solar System Facts', 'page': 2, 'to

In [40]:
# -------------------------
# Imports
# -------------------------
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableMap, RunnablePassthrough
from langchain_openai import ChatOpenAI

# -------------------------
# Step 1: LLM
# -------------------------
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# -------------------------
# Step 2: Retriever
# -------------------------
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# -------------------------
# Step 3: Prompt
# -------------------------
#system_prompt = """You are a strict question-answering assistant.

#You must ONLY use the provided context to answer the question.
#If the answer is NOT explicitly present in the context, reply with:
#"I don't know based on the provided context."

#Do NOT use prior knowledge.
#Do NOT guess.
#Do NOT make up answers.

#Context:
#{context}
#"""
system_prompt = """You are a question-answering assistant.

Use ONLY the provided context to answer the question.

You are allowed to:
- Make reasonable inferences from the context
- Use relationships between entities mentioned in the context
  (for example, if something is described as part of a country,
   you can use it to describe that country)

If the answer cannot be reasonably derived from the context, say:
"I don't know based on the provided context."

Do NOT use outside knowledge.

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{query}")
    ]
)

# -------------------------
# Step 4: Format documents
# -------------------------
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into the prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        #print("\n---DOC---\n", doc.page_content)
        source = doc.metadata.get("source", 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}) ---\n{doc.page_content}")
    return "\n\n".join(formatted)


def debug_format_docs(docs):
    for d in docs:
        print("\n---DOC---\n", d.page_content)
    return "\n\n".join(doc.page_content for doc in docs)
# -------------------------
# Step 5: Build RAG pipeline
# -------------------------
simple_rag_chain = (
    RunnableMap({
        "context": retriever | format_docs,   # retrieve + format
        "query": RunnablePassthrough()        # pass question as-is
    })
    | prompt
    | llm
    | StrOutputParser()                    # extract text from LLM response
)
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001DD4D7C3F10>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  query: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are a question-answering assistant.\n\nUse ONLY the provided context to answer the question.\n\nYou are allowed to:\n- Make reasonable inferences from the context\n- Use relationships between entities mentioned in the context\n  (for example, if something is described as part of a country,\n   you can use it to describe that country)\n\nIf the answer cannot be reasonably derived from the context, say:\n"I don\'t know based on the provided context."\n\nDo NOT use outside knowledge.\n\nContext:\n{cont

### Conversational RAG Chain

In [74]:
import faiss
import numpy as np
documents = [
    "The Eiffel Tower is in Paris and is a famous landmark.",
    "The Louvre Museum has the Mona Lisa painting.",
    "France has a rich history of art and culture."
]

docs = [Document(page_content=d) for d in documents]

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=len, separators=[" "])
docs = text_splitter.split_documents(docs)
print("Formatted documents for prompt:")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)

# Extract text using .page_content
doc_texts = [doc.page_content for doc in docs]
doc_vectors = embeddings.embed_documents(doc_texts)

dim = len(doc_vectors[0])
index = faiss.IndexFlatL2(dim)  # L2 distance
index.add(np.array(doc_vectors, dtype=np.float32))

# Store mapping to original texts
doc_mapping = {i: doc_texts[i] for i in range(len(doc_texts))}

# Define Retrieval Function
def retrieve(query, k=2):
    query_vector = np.array(embeddings.embed_query(query), dtype=np.float32).reshape(1, -1)
    distances, indices = index.search(query_vector, k)
    results = [doc_mapping[i] for i in indices[0]]
    return results

# Conversational RAG
conversation_history = []

def chat(query):
    # Retrieve relevant docs
    context_docs = retrieve(query)
    context_text = "\n".join(context_docs)
    
    # Combine conversation history
    history_text = "\n".join([f"User: {u}\nAI: {a}" for u, a in conversation_history])
    
    # Prompt for LLM
    prompt = f"{history_text}\nUser: {query}\nRelevant Information:\n{context_text}\nAI:"
    
    response = llm.invoke(prompt)
    answer = response.content
    
    print(conversation_history)
    # Update conversation history
    conversation_history.append((query, answer))
    
    return answer

print(chat("Where is the Mona Lisa located?"))
print(chat("Tell me something about Paris landmarks."))
print(chat("What can you say about French art?"))
print(chat("What is the capital of France?"))
print(chat("What is the largest planet in our solar system?"))

Formatted documents for prompt:
[]
The Mona Lisa is located in the Louvre Museum in Paris, France.
[('Where is the Mona Lisa located?', 'The Mona Lisa is located in the Louvre Museum in Paris, France.')]
Paris is home to many famous landmarks, including the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral, and the Arc de Triomphe. The city is known for its beautiful architecture, world-class museums, and vibrant cultural scene. Paris is also famous for its delicious cuisine, fashion, and romantic atmosphere.
[('Where is the Mona Lisa located?', 'The Mona Lisa is located in the Louvre Museum in Paris, France.'), ('Tell me something about Paris landmarks.', 'Paris is home to many famous landmarks, including the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral, and the Arc de Triomphe. The city is known for its beautiful architecture, world-class museums, and vibrant cultural scene. Paris is also famous for its delicious cuisine, fashion, and romantic atmosphere.')]
French art h

In [86]:
# Example documents
docs = [
    Document(page_content="The Eiffel Tower is in Paris and is a famous landmark."),
    Document(page_content="The Louvre Museum has the Mona Lisa painting."),
    Document(page_content="France has a rich history of art and culture."),
]

# Split into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
split_docs = text_splitter.split_documents(docs)


embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)

# Create FAISS index
vectorstore = FAISS.from_documents(split_docs, embeddings)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})

# OpenAI chat model
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# Conversational RAG chain
# Store conversation history
chat_history = []

def chat(query):
    # Use retriever to get top-k relevant documents
    relevant_docs = retriever.get_relevant_documents(query) if hasattr(retriever, "get_relevant_documents") else vectorstore.similarity_search(query, k=2)
    
    context = "\n".join([doc.page_content for doc in relevant_docs])
    
    # Include previous chat history in the prompt
    history_text = "\n".join([f"User: {q}\nAI: {a}" for q, a in chat_history])
    prompt = f"""
    You are an AI assistant. Answer the question ONLY using the context below.
    If the answer is not in the context, respond with "I don't know".

    Context:
    {context}

    Question: {query}
    Answer:
    """
    
    # Generate answer
    response = llm.invoke(prompt)
    answer = response.content if hasattr(response, "content") else response
    
    print(chat_history)
    # Update chat history
    chat_history.append((query, answer))
    return answer
print(chat("Where is the Mona Lisa located?"))
print(chat("Tell me something about Paris landmarks."))
print(chat("What is Artificial Intelligence?"))
print(chat("What is the capital of France?"))
print(chat("What is the largest planet in our solar system?"))

[]
The Mona Lisa is located in The Louvre Museum.
[('Where is the Mona Lisa located?', 'The Mona Lisa is located in The Louvre Museum.')]
The Eiffel Tower is in Paris.
[('Where is the Mona Lisa located?', 'The Mona Lisa is located in The Louvre Museum.'), ('Tell me something about Paris landmarks.', 'The Eiffel Tower is in Paris.')]
I don't know.
[('Where is the Mona Lisa located?', 'The Mona Lisa is located in The Louvre Museum.'), ('Tell me something about Paris landmarks.', 'The Eiffel Tower is in Paris.'), ('What is Artificial Intelligence?', "I don't know.")]
Paris
[('Where is the Mona Lisa located?', 'The Mona Lisa is located in The Louvre Museum.'), ('Tell me something about Paris landmarks.', 'The Eiffel Tower is in Paris.'), ('What is Artificial Intelligence?', "I don't know."), ('What is the capital of France?', 'Paris')]
I don't know.


### Streaming RAG Chain

In [41]:
streaming_rag_chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | prompt
    | llm
)

print("Streaming RAG chain created successfully:")

streaming_rag_chain

Streaming RAG chain created successfully:


{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001DD4D7C3F10>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  query: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are a question-answering assistant.\n\nUse ONLY the provided context to answer the question.\n\nYou are allowed to:\n- Make reasonable inferences from the context\n- Use relationships between entities mentioned in the context\n  (for example, if something is described as part of a country,\n   you can use it to describe that country)\n\nIf the answer cannot be reasonably derived from the context, say:\n"I don\'t know based on the provided context."\n\nDo NOT use outside knowledge.\n\nContext:\n{cont

Available Chains:
- Simple_RAG_Chain -> Basic Q&A
- Conversational_RAG_Chain -> Maintains conversational history
- Streaming_RAG_Chain -> Supports Token Streaming


In [42]:
# -------------------------
# Step 6: Run
# -------------------------
question = "What is Deep Learning?"

response = simple_rag_chain.invoke(question)

print("Question:", question)
print("Answer:", response)

Question: What is Deep Learning?
Answer: I don't know based on the provided context.


In [ ]:
question = "What is France?"

response = conversational_rag.invoke({"query": question})
print("Question:", question)
print("Answer:", response)

In [43]:
question = "Tell me about Solar System?"

print("Streaming RAG")
print("Answer:", end=' ', flush=True)
for chunk in streaming_rag_chain.stream(question):
     print(chunk.content, end='', flush=True)
print()

Streaming RAG
Answer: The Solar System consists of the Sun and all the celestial objects that orbit around it, including planets, moons, asteroids, comets, and other space debris. Jupiter is the largest planet in our Solar System and is classified as a gas giant with a strong magnetic field. Jupiter has at least 79 moons, with the four largest known as the Galilean moons: Io, Europa, Ganymede, and Callisto.


In [44]:
def test_rag_chain(question: str):
    """Test all RAG chain variants"""
    print("Question:", question)
    print("=" * 80)
    
    # 1. Simple RAG
    print("\n--- Simple RAG ---")
    answer = simple_rag_chain.invoke(question)
    print("Answer:", answer)

    # 2. Streaming RAG
    print("\n--- Streaming RAG ---")
    
    print("Answer:", end=' ', flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end='', flush=True)
    print()

In [45]:
test_rag_chain("What is the capital of France?")

Question: What is the capital of France?

--- Simple RAG ---
Answer: The capital of France is Paris.

--- Streaming RAG ---
Answer: The capital of France is Paris.


In [46]:
# Test with multiple questions
questions = [
    "What is the capital of France?",
    "What is the largest planet in our solar system?",
    "What is the chemical symbol for water?"
]  
for question in questions:
    print("\n" + "="*80 + "\n")
    test_rag_chain(question)



Question: What is the capital of France?

--- Simple RAG ---
Answer: The capital of France is Paris.

--- Streaming RAG ---
Answer: The capital of France is Paris.


Question: What is the largest planet in our solar system?

--- Simple RAG ---
Answer: The largest planet in our solar system is Jupiter.

--- Streaming RAG ---
Answer: The largest planet in our solar system is Jupiter.


Question: What is the chemical symbol for water?

--- Simple RAG ---
Answer: The chemical symbol for water is H2O.

--- Streaming RAG ---
Answer: The chemical symbol for water is H2O.
